

Context managers guide · MD
# Python Context Managers: Beginner se Advanced (Hinglish)
 
> Diagrams ke liye: GitHub, VS Code (Markdown Preview Mermaid Support extension), Obsidian ya Notion jaise viewers mein ye file kholo. Har node ka fill aur text color explicitly set hai, isliye **light aur dark dono theme** mein clearly dikhte hain.
 
## Table of Contents
 
1. [Problem kya hai?](#part-1-problem-kya-hai-beginner)
2. [`__enter__` / `__exit__`](#part-2-__enter__-aur-__exit__)
3. [Exceptions aur suppress](#part-3-exceptions-handle-karna)
4. [`@contextmanager`](#part-4-contextmanager-sabse-easy-tarika)
5. [Real-life examples](#part-5-real-life-examples)
6. [Stdlib ke ready-made managers](#part-6-standard-library-ke-ready-made-managers)
7. [Multiple managers aur `ExitStack`](#part-7-multiple-managers-aur-exitstack-advanced)
8. [`ContextDecorator`, reusable, reentrant](#part-8-contextdecorator-reusable-reentrant-advanced)
9. [Async context managers](#part-9-async-context-managers-advanced)
10. [Mistakes, cheat sheet, practice](#part-10-mistakes-cheat-sheet-practice)
---
 
## Part 1: Problem kya hai? (Beginner)
 
File open karte ho to kaam ke baad **close** karna zaroori hai. Agar beech mein error aa gaya to `close()` chalega hi nahi, aur resource leak ho jayega.
 
**Risky tarika:**
 
```python
f = open("a.txt", "w")
f.write("hello")   # yahan error aaya to...
f.close()          # ...ye line kabhi nahi chalegi
```
 
**Thoda better, lekin lamba:**
 
```python
f = open("a.txt", "w")
try:
    f.write("hello")
finally:
    f.close()
```
 
**Best: `with` statement:**
 
```python
with open("a.txt", "w") as f:
    f.write("hello")
# yahan file AUTOMATIC close ho chuki hai, error aaye ya na aaye
```
 
> **Context manager = "setup + cleanup ka guaranteed jodi".**
> Setup `with` block se pehle, cleanup block ke baad, error ho to bhi.
 
```mermaid
%%{init: {'theme':'base','themeVariables':{'fontSize':'15px'}}}%%
flowchart LR
    A([with shuru]):::start --> B[Setup<br/>resource open]:::setup
    B --> C[Tumhara code<br/>with block ke andar]:::work
    C --> D[Cleanup<br/>resource close]:::clean
    C -. error aaya .-> D
    D --> E([with khatam]):::done
 
    classDef start fill:#FFD166,stroke:#B8860B,stroke-width:2px,color:#1a1a1a
    classDef setup fill:#06D6A0,stroke:#04856a,stroke-width:2px,color:#0b2e25
    classDef work  fill:#74C0FC,stroke:#1971C2,stroke-width:2px,color:#0b2540
    classDef clean fill:#FF8787,stroke:#C92A2A,stroke-width:2px,color:#3a0a0a
    classDef done  fill:#B197FC,stroke:#6741D9,stroke-width:2px,color:#1e0f4d
```
 
**Example: error ke baad bhi file close hoti hai**
 
```python
with open("demo.txt", "w") as f:
    f.write("Namaste, context manager!")
print(f.closed)   # True
 
try:
    with open("demo.txt") as f:
        raise ValueError("kuch galat ho gaya")
except ValueError:
    pass
print(f.closed)   # True, error ke baad bhi band hui
```
 
---
 
## Part 2: `__enter__` aur `__exit__`
 
Koi bhi object jisme `__enter__` aur `__exit__` methods ho, wo context manager hai.
 
```python
with Manager() as x:
    ...
```
 
Andar se Python ye karta hai:
 
```python
mgr = Manager()
x = mgr.__enter__()      # setup, return value x mein jaati hai
try:
    ...                  # tumhara block
finally:
    mgr.__exit__(...)    # cleanup, hamesha chalta hai
```
 
```mermaid
%%{init: {'theme':'base'}}%%
sequenceDiagram
    autonumber
    participant U as Tumhara Code
    participant M as Context Manager
    rect rgb(255, 236, 179)
    U->>M: with Manager() as x
    M-->>U: __enter__() se x return
    end
    rect rgb(187, 222, 251)
    Note over U: with block ka code chalta hai
    end
    rect rgb(255, 205, 210)
    U->>M: __exit__(exc_type, exc_val, tb)
    M-->>U: False/None = error aage jaane do<br/>True = error dba do (suppress)
    end
```
 
**Example: class-based manager**
 
```python
class Greeter:
    def __init__(self, name):
        self.name = name
 
    def __enter__(self):
        print(f"[enter] Namaste {self.name}! (setup)")
        return self              # `as g` mein yahi milega
 
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"[exit ] Alvida {self.name}! (cleanup)")
        return False             # exception ko dabana nahi
 
with Greeter("Samarth") as g:
    print("[body ] andar ka kaam, name =", g.name)
```
 
Output:
 
```
[enter] Namaste Samarth! (setup)
[body ] andar ka kaam, name = Samarth
[exit ] Alvida Samarth! (cleanup)
```
 
---
 
## Part 3: Exceptions handle karna
 
`__exit__(self, exc_type, exc_val, exc_tb)` ke 3 arguments:
 
- Block mein error **nahi** aaya: teeno `None`.
- Error aaya: type, instance, traceback mil jaate hain.
Return value ka matlab:
 
| Return | Matlab |
|---|---|
| `True` | Error **suppress** (dab gaya, program aage chalega) |
| `False` / `None` | Error dobara raise hoga (normal behaviour) |
 
```mermaid
%%{init: {'theme':'base'}}%%
flowchart TD
    S([Block ke andar kaam]):::start --> Q{Error aaya?}:::decision
    Q -- Nahi --> E1["__exit__(None, None, None)"]:::ok
    Q -- Haan --> E2["__exit__(type, val, tb)"]:::err
    E2 --> R{Return value?}:::decision
    R -- True --> SUP[Error suppress<br/>code aage chalega]:::ok
    R -- False/None --> RAI[Error dobara raise<br/>program ruk sakta hai]:::bad
    E1 --> END([Done]):::done
    SUP --> END
 
    classDef start fill:#FFD166,stroke:#B8860B,stroke-width:2px,color:#1a1a1a
    classDef decision fill:#FFA94D,stroke:#D9480F,stroke-width:2px,color:#2b1100
    classDef ok fill:#69DB7C,stroke:#2B8A3E,stroke-width:2px,color:#0a2e12
    classDef err fill:#FFA8A8,stroke:#C92A2A,stroke-width:2px,color:#3a0a0a
    classDef bad fill:#FF6B6B,stroke:#8B0000,stroke-width:2px,color:#ffffff
    classDef done fill:#B197FC,stroke:#6741D9,stroke-width:2px,color:#1e0f4d
```
 
**Example: sirf ZeroDivisionError suppress karo**
 
```python
class IgnoreZeroDivision:
    def __enter__(self):
        return self
 
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is ZeroDivisionError:
            print(f"[exit ] Dab gaya: {exc_val}")
            return True       # suppress
        return False          # baaki errors normal raise
 
with IgnoreZeroDivision():
    print(10 / 0)
print("Program aage chal raha hai!")
 
try:
    with IgnoreZeroDivision():
        int("abc")            # ValueError suppress NAHI hoga
except ValueError:
    print("ValueError bahar aa gaya")
```
 
---
 
## Part 4: `@contextmanager` (sabse easy tarika)
 
Class likhna lamba lagta hai? **Generator function + `@contextmanager`** use karo.
 
```python
from contextlib import contextmanager
 
@contextmanager
def my_cm():
    # --- SETUP ---
    yield value        # <- yahan with block chalta hai
    # --- CLEANUP ---
```
 
> **Golden rule:** cleanup hamesha `try/finally` mein rakho, warna error aane par cleanup skip ho jayega.
 
```mermaid
%%{init: {'theme':'base'}}%%
flowchart LR
    A[with my_cm as v]:::start --> B[Setup code<br/>yield se pehle]:::setup
    B --> C{{yield value}}:::yield
    C --> D[with block<br/>ka code]:::work
    D --> E[finally:<br/>cleanup code]:::clean
    D -. error .-> E
    E --> F([Done]):::done
 
    classDef start fill:#FFD166,stroke:#B8860B,stroke-width:2px,color:#1a1a1a
    classDef setup fill:#63E6BE,stroke:#0CA678,stroke-width:2px,color:#053d2f
    classDef yield fill:#FFA94D,stroke:#D9480F,stroke-width:3px,color:#2b1100
    classDef work fill:#74C0FC,stroke:#1971C2,stroke-width:2px,color:#0b2540
    classDef clean fill:#FF8787,stroke:#C92A2A,stroke-width:2px,color:#3a0a0a
    classDef done fill:#B197FC,stroke:#6741D9,stroke-width:2px,color:#1e0f4d
```
 
**Example 1: basic**
 
```python
@contextmanager
def my_cm(label):
    print(f"[setup  ] {label} shuru")
    try:
        yield label.upper()       # `as v` mein ye value jayegi
    finally:
        print(f"[cleanup] {label} khatam")
 
with my_cm("demo") as v:
    print("[body   ] value mili:", v)
```
 
**Example 2: error ke saath bhi cleanup chalta hai**
 
```python
try:
    with my_cm("risky"):
        raise RuntimeError("boom")
except RuntimeError:
    print("Error bahar aaya, par cleanup ho chuka tha")
```
 
**Example 3: generator mein error catch karke suppress**
 
```python
@contextmanager
def swallow_errors(*exc_types):
    try:
        yield
    except exc_types as e:
        print(f"[swallowed] {type(e).__name__}: {e}")
 
with swallow_errors(KeyError):
    {}["missing"]
print("Program aage chal raha hai")
```
 
---
 
## Part 5: Real-life examples
 
Pattern yaad rakho: **"kuch badlo, kaam karo, wapas purana kar do"**.
 
```mermaid
%%{init: {'theme':'base'}}%%
mindmap
  root((Context<br/>Manager<br/>use cases))
    Files
      open / close
      temp files
    Time
      timer
      timeout
    State
      chdir
      env vars
      decimal precision
    Safety
      locks
      DB transaction
      rollback
    Output
      redirect stdout
      logging capture
```
 
### 5.1 Timer
 
```python
import time
from contextlib import contextmanager
 
@contextmanager
def timer(name="block"):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"[timer] {name}: {time.perf_counter() - start:.4f}s")
 
with timer("sum of 1M numbers"):
    sum(range(1_000_000))
```
 
### 5.2 Temporary directory change
 
```python
import os
 
@contextmanager
def change_dir(path):
    old = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old)          # hamesha wapas original par
 
with change_dir("/tmp"):
    print("Andar:", os.getcwd())
print("Baahar:", os.getcwd())
```
 
### 5.3 Temporary environment variables
 
```python
@contextmanager
def temp_env(**kwargs):
    old = {k: os.environ.get(k) for k in kwargs}
    os.environ.update({k: str(v) for k, v in kwargs.items()})
    try:
        yield
    finally:
        for k, v in old.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v
 
with temp_env(MY_MODE="debug"):
    print(os.environ["MY_MODE"])       # debug
print(os.environ.get("MY_MODE"))       # None
```
 
### 5.4 Database transaction (commit / rollback)
 
```python
class FakeDB:
    def __init__(self):
        self.data = {}
 
    @contextmanager
    def transaction(self):
        snapshot = dict(self.data)     # backup
        try:
            yield self
            print("[db] COMMIT")
        except Exception:
            self.data = snapshot       # rollback
            print("[db] ROLLBACK")
            raise
 
db = FakeDB()
with db.transaction():
    db.data["user"] = "Samarth"
print(db.data)                          # {'user': 'Samarth'}
 
try:
    with db.transaction():
        db.data["user"] = "Hacker"
        raise ValueError("payment fail")
except ValueError:
    pass
print(db.data)                          # {'user': 'Samarth'}, rollback ho gaya
```
 
---
 
## Part 6: Standard library ke ready-made managers
 
| Tool | Kaam |
|---|---|
| `suppress(Exc)` | `try/except/pass` ka short form |
| `redirect_stdout(f)` | `print()` ka output kisi aur jagah bhejo |
| `closing(obj)` | Jis object mein `.close()` ho use auto-close karo |
| `nullcontext(x)` | "Kuch nahi karne wala" manager (conditional use) |
| `tempfile.TemporaryDirectory()` | Temp folder, auto delete |
| `threading.Lock()` | `with lock:` se safe locking |
 
```python
import io, os, tempfile
from contextlib import suppress, redirect_stdout, closing, nullcontext
 
# suppress
with suppress(FileNotFoundError):
    os.remove("file_jo_exist_nahi_karti.txt")
 
# redirect_stdout: print ka output string mein pakdo
buf = io.StringIO()
with redirect_stdout(buf):
    print("ye screen par nahi aayega")
print(repr(buf.getvalue()))
 
# closing
class Resource:
    def close(self):
        print("[Resource] closed")
 
with closing(Resource()):
    print("Resource use ho raha hai")
 
# nullcontext: optional manager
use_timer = False
with (timer("optional") if use_timer else nullcontext()):
    print("Timer ke bina chala")
 
# TemporaryDirectory
with tempfile.TemporaryDirectory() as d:
    print(os.path.exists(d))    # True
print(os.path.exists(d))        # False
```
 
---
 
## Part 7: Multiple managers aur `ExitStack` (Advanced)
 
```python
# Comma se
with open("a") as f1, open("b") as f2:
    ...
 
# Python 3.10+: parentheses ke saath
with (
    open("a") as f1,
    open("b") as f2,
):
    ...
```
 
**Cleanup order ulta hota hai (LIFO):** jo last mein khula wo pehle band hoga.
 
```mermaid
%%{init: {'theme':'base'}}%%
flowchart TB
    subgraph ENTER [Enter order: upar se neeche]
        direction TB
        A1[1. File A open]:::a --> A2[2. File B open]:::b --> A3[3. File C open]:::c
    end
    subgraph EXIT [Exit order: ulta LIFO]
        direction TB
        B1[3. File C close]:::c --> B2[2. File B close]:::b --> B3[1. File A close]:::a
    end
    A3 ==> B1
 
    classDef a fill:#74C0FC,stroke:#1971C2,stroke-width:2px,color:#0b2540
    classDef b fill:#63E6BE,stroke:#0CA678,stroke-width:2px,color:#053d2f
    classDef c fill:#FFD43B,stroke:#E67700,stroke-width:2px,color:#3a2a00
```
 
Jab managers ki ginti **runtime par decide** ho (loop mein), `ExitStack` use karo:
 
```python
from contextlib import ExitStack, contextmanager
 
@contextmanager
def tracked(name):
    print(f"  enter {name}")
    try:
        yield name
    finally:
        print(f"  exit  {name}")
 
# Nested
with tracked("A"), tracked("B"), tracked("C"):
    print("  ...kaam chal raha hai")
# exit C, exit B, exit A
 
# ExitStack (dynamic count)
with ExitStack() as stack:
    for n in ["X", "Y", "Z"]:
        stack.enter_context(tracked(n))
    stack.callback(print, "  [callback] custom cleanup, sabse pehle chalega")
    print("  ...sab khul gaye")
```
 
Real use: `files = [stack.enter_context(open(p)) for p in paths]`, kitni bhi files ek saath, sab safely band hongi.
 
---
 
## Part 8: `ContextDecorator`, reusable, reentrant (Advanced)
 
### 8.1 ContextDecorator: manager ko decorator ki tarah bhi use karo
 
```python
from contextlib import ContextDecorator
 
class Logged(ContextDecorator):
    def __init__(self, label):
        self.label = label
 
    def __enter__(self):
        print(f"  >> {self.label} start")
        return self
 
    def __exit__(self, *exc):
        print(f"  << {self.label} end")
        return False
 
@Logged("calculate")
def calculate(a, b):
    return a + b
 
calculate(2, 3)            # decorator ki tarah
 
with Logged("manual"):     # with ki tarah
    pass
```
 
### 8.2 Single-use vs Reusable vs Reentrant
 
```mermaid
%%{init: {'theme':'base'}}%%
flowchart LR
    CM[Context Manager<br/>types]:::root --> S[Single-use<br/>@contextmanager]:::single
    CM --> R[Reusable<br/>class, no state]:::reuse
    CM --> RE[Reentrant<br/>RLock, nested OK]:::reent
    S --> S1[dobara use<br/>= Error]:::bad
    R --> R1[baar baar<br/>with chalega]:::good
    RE --> RE1[with ke andar<br/>with chalega]:::good
 
    classDef root fill:#FFD166,stroke:#B8860B,stroke-width:2px,color:#1a1a1a
    classDef single fill:#FFA8A8,stroke:#C92A2A,stroke-width:2px,color:#3a0a0a
    classDef reuse fill:#74C0FC,stroke:#1971C2,stroke-width:2px,color:#0b2540
    classDef reent fill:#B197FC,stroke:#6741D9,stroke-width:2px,color:#1e0f4d
    classDef good fill:#69DB7C,stroke:#2B8A3E,stroke-width:2px,color:#0a2e12
    classDef bad fill:#FF6B6B,stroke:#8B0000,stroke-width:2px,color:#ffffff
```
 
```python
cm = my_cm("once")
with cm:
    pass
with cm:        # Error! @contextmanager wala object sirf ek baar use hota hai
    pass        # (error type Python version par depend karta hai)
 
lg = Logged("reuse-test")
with lg: pass
with lg: pass   # class-based, bina state, baar baar chalega
```
 
---
 
## Part 9: Async context managers (Advanced)
 
Jab setup/cleanup mein `await` chahiye (network, async DB), tab:
 
| Sync | Async |
|---|---|
| `__enter__` / `__exit__` | `__aenter__` / `__aexit__` |
| `@contextmanager` | `@asynccontextmanager` |
| `with` | `async with` |
 
```mermaid
%%{init: {'theme':'base'}}%%
sequenceDiagram
    participant C as async function
    participant M as Async Manager
    rect rgb(200, 247, 220)
    C->>M: async with M() as x
    M-->>C: await __aenter__()
    end
    rect rgb(255, 236, 179)
    Note over C: await wala kaam
    end
    rect rgb(255, 205, 210)
    C->>M: await __aexit__()
    M-->>C: cleanup done
    end
```
 
```python
import asyncio, time, contextlib
 
class AsyncConnection:
    async def __aenter__(self):
        await asyncio.sleep(0.05)       # fake connect
        print("  [async] connected")
        return self
 
    async def __aexit__(self, exc_type, exc, tb):
        await asyncio.sleep(0.05)       # fake disconnect
        print("  [async] disconnected")
        return False
 
    async def query(self, q):
        await asyncio.sleep(0.01)
        return f"result of '{q}'"
 
@contextlib.asynccontextmanager
async def async_timer(name):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"  [async_timer] {name}: {time.perf_counter() - start:.2f}s")
 
async def main():
    async with async_timer("db work"):
        async with AsyncConnection() as conn:
            print(await conn.query("SELECT 1"))
 
asyncio.run(main())
```
 
---
 
## Part 10: Mistakes, cheat sheet, practice
 
### Common mistakes
 
1. `@contextmanager` mein `yield` ke baad cleanup **bina `try/finally`** likhna, error aaya to cleanup skip.
2. `__exit__` mein galti se `return True` likh dena, saare errors chup-chaap dab jaate hain (debugging nightmare).
3. `@contextmanager` wale object ko dobara `with` mein use karna.
4. `yield` ek se zyada baar karna (sirf **ek** yield allowed hai).
5. `with open(...) as f` ke baad `f` ko bahar use karna (wo closed hoga).
**Mistake demo: try/finally ke bina**
 
```python
@contextmanager
def bad():
    print("  setup")
    yield
    print("  cleanup")      # error aaya to ye KABHI nahi chalega
 
try:
    with bad():
        raise ValueError("oops")
except ValueError:
    pass
```
 
### Cheat sheet
 
| Situation | Kya use karein |
|---|---|
| Simple setup/cleanup | `@contextmanager` |
| State / reuse chahiye | Class (`__enter__` / `__exit__`) |
| Error ignore karna | `contextlib.suppress` |
| Kai managers, dynamic | `ExitStack` |
| Decorator bhi chahiye | `ContextDecorator` |
| `await` chahiye | `async with` / `@asynccontextmanager` |
| Optional manager | `nullcontext` |
 
```mermaid
%%{init: {'theme':'base'}}%%
flowchart TD
    Q1{Async kaam<br/>hai?}:::q -- Haan --> AS[asynccontextmanager<br/>ya __aenter__]:::async
    Q1 -- Nahi --> Q2{Managers ki ginti<br/>runtime par decide?}:::q
    Q2 -- Haan --> ES[ExitStack]:::stack
    Q2 -- Nahi --> Q3{Complex state<br/>ya reuse chahiye?}:::q
    Q3 -- Haan --> CL[Class based<br/>__enter__/__exit__]:::cls
    Q3 -- Nahi --> GEN["@contextmanager"]:::gen
 
    classDef q fill:#FFD166,stroke:#B8860B,stroke-width:2px,color:#1a1a1a
    classDef async fill:#B197FC,stroke:#6741D9,stroke-width:2px,color:#1e0f4d
    classDef stack fill:#74C0FC,stroke:#1971C2,stroke-width:2px,color:#0b2540
    classDef cls fill:#FF8787,stroke:#C92A2A,stroke-width:2px,color:#3a0a0a
    classDef gen fill:#69DB7C,stroke:#2B8A3E,stroke-width:2px,color:#0a2e12
```
 
### Practice (pehle khud try karo, phir compare karo)
 
1. `cd_temp()` banao jo temp folder banaye, usme `chdir` kare, aur exit par wapas original folder aaye + temp folder delete ho.
2. `Counter` class banao jo `with` ke andar kitni baar `.hit()` call hua count kare, aur exit par total print kare.
3. Ek manager jo `sys.stdout` ko temporarily file mein bheje, `redirect_stdout` use kiye **bina**.
4. Soch ke batao: `retry(times=3)` context manager ban sakta hai? (Hint: `@contextmanager` mein sirf ek `yield` hota hai. Isliye retry ke liye decorator ya loop kyun behtar hai?)
 

